# Mejoras California Housing — Guía de Examen

## ¿Por qué este proyecto es probable en el examen?
El dataset está en el propio repositorio (`data/housing.csv`), así que **funciona sin internet**.
Es el ejemplo canónico de regresión end-to-end del libro de Aurélien Géron.

## Tipo de problema
**Regresión supervisada** — predecir `median_house_value` (precio medio de vivienda).

## Métricas objetivo
| Métrica | Qué mide | Cuándo usarla |
|---------|----------|---------------|
| RMSE | Error medio en las mismas unidades que el target | Siempre en regresión |
| MAE | Error absoluto medio, más robusto a outliers | Si hay outliers extremos |
| R² | % de varianza explicada (1.0 = perfecto) | Para comparar modelos |

## Mejoras clave vs. proyecto base
1. **Ratios** en vez de totales → evita que "casas con más habitaciones = más caras" sea espurio
2. **Log-transform** en variables sesgadas → distribuciones más normales, mejor rendimiento de modelos lineales
3. **Clustering geográfico** con KMeans + RBF kernel → captura el efecto de la ubicación sin necesidad de conocer el código postal
4. **Pipeline sin leakage** → el `fit_transform` SOLO sobre train
5. **RandomizedSearchCV** → búsqueda eficiente de hiperparámetros


## Paso 1 — Cargar datos y split estratificado

**Por qué estratificamos por `income_cat`?**
La renta media (`median_income`) es el predictor más correlacionado con el precio.
Si hacemos un split aleatorio puro, por mala suerte el test podría tener mucha más renta alta que el train,
y nuestras métricas de test serían engañosas.
La estratificación garantiza que la distribución de renta sea la misma en train y test.

**Por qué borramos `income_cat` después del split?**
Era una columna auxiliar que creamos SOLO para estratificar. No existe en el dataset original
y no queremos que el modelo la use.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Rutas robustas: funciona tanto desde proyects-upgrades/ como desde cualquier nivel
DATA_PATH = Path("../proyects/COMPLETOS/california-e2e/data/housing.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/housing.csv")

df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(df.head(3))


In [ ]:
# Crear categoría de income SOLO para estratificar
df["income_cat"] = pd.cut(
    df["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5],
)

train_set, test_set = train_test_split(
    df,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df["income_cat"],   # ← clave: distribución representativa
)

# Borrar la columna auxiliar en ambos splits
for split in (train_set, test_set):
    split.drop(columns=["income_cat"], inplace=True)

X_train = train_set.drop(columns=["median_house_value"])
y_train = train_set["median_house_value"]
X_test  = test_set.drop(columns=["median_house_value"])
y_test  = test_set["median_house_value"]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


## Paso 2 — Transformador de Clustering Geográfico

**Por qué no usamos lat/lon directamente?**
La latitud y longitud son coordenadas numéricas pero su relación con el precio **no es lineal**.
San Francisco y Los Ángeles están en latitudes distintas y tienen precios distintos, pero no por su latitud.
Lo que importa es la "zona" a la que pertenece una casa.

**Cómo funciona `ClusterSimilarity`?**
1. `fit`: aplica KMeans sobre las coordenadas para encontrar `n_clusters` zonas geográficas
2. `transform`: para cada casa, devuelve su similitud (kernel RBF) con cada centro de cluster
   → una casa "en San Francisco" tendrá similitud alta al cluster de SF y baja al de LA

**Por qué hereda de `BaseEstimator, TransformerMixin`?**
Así puede usarse dentro de un Pipeline de sklearn y tiene los métodos `get_params`/`set_params`
que necesita `RandomizedSearchCV` para ajustar `n_clusters` y `gamma`.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import rbf_kernel

class ClusterSimilarity(BaseEstimator, TransformerMixin):
    """
    Convierte lat/lon en features de similitud geográfica.
    n_clusters: número de zonas geográficas
    gamma: anchura del kernel RBF (mayor gamma = similitud cae más rápido con la distancia)
    """
    def __init__(self, n_clusters=10, gamma=1.0, random_state=42):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None):
        self.kmeans_ = KMeans(
            n_clusters=self.n_clusters,
            n_init=10,            # 10 inicializaciones aleatorias, escoge la mejor
            random_state=self.random_state
        )
        self.kmeans_.fit(X)
        return self

    def transform(self, X):
        # Devuelve matriz (n_muestras, n_clusters) con similitudes
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)


## Paso 3 — Pipeline de Preprocesamiento

**Por qué crear pipelines separados para ratios, logs y categorías?**
Cada grupo de variables necesita una transformación diferente.
`ColumnTransformer` aplica cada sub-pipeline SOLO a las columnas indicadas,
manteniendo el código ordenado y sin mezclar transformaciones.

**Por qué usamos ratios y no los valores absolutos?**
- `total_bedrooms / total_rooms` (porcentaje de habitaciones que son dormitorios) es comparable entre edificios de distintos tamaños
- `total_rooms / households` (habitaciones por hogar) indica densidad real
- `population / households` (personas por hogar) indica hacinamiento
Los totales brutos correlacionan con el tamaño del distrito, que no es lo que queremos predecir.

**Por qué log en `total_bedrooms`, `total_rooms`, `population`, `households`, `median_income`?**
Estas variables tienen distribución muy sesgada (cola derecha larga).
El log las aproxima a distribuciones normales, mejorando modelos lineales y neurales.
RandomForest no lo necesita, pero tampoco le hace daño.

**Por qué `remainder=Pipeline([imputer, scaler])` y no `remainder='drop'`?**
Queremos mantener las features numéricas que no están explícitamente en ningún grupo
(como `housing_median_age`, `median_income` sin log) pero escaladas correctamente.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

# Función para calcular ratios — opera sobre arrays numpy
def ratio(X):
    return X[:, [0]] / X[:, [1]]     # columna_0 / columna_1

def ratio_names(transformer, feature_names_in):
    return ["ratio"]                  # nombre de la feature resultante

# Pipeline para ratios: imputa → calcula ratio → escala
ratio_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),   # NaN = mediana antes de dividir
    ("ratio",   FunctionTransformer(ratio, feature_names_out=ratio_names)),
    ("scaler",  StandardScaler()),
])

# Pipeline para variables sesgadas: imputa → log1p → escala
# log1p = log(1+x), funciona con ceros (log(0) no existe pero log(1+0)=0)
log_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log",     FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler",  StandardScaler()),
])

# Pipeline para categorías: imputa → one-hot encoding
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),   # ignora categorías no vistas en train
])

# Ensamblamos todo en un ColumnTransformer
preprocessor = ColumnTransformer([
    ("bedrooms_ratio", ratio_pipeline, ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline, ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline, ["population", "households"]),
    ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                           "households", "median_income"]),
    ("geo", ClusterSimilarity(random_state=RANDOM_STATE), ["latitude", "longitude"]),
    ("cat", cat_pipeline, ["ocean_proximity"]),
],
# Las columnas no mencionadas (housing_median_age, latitude, longitude si no la tratáramos...)
# se imputan y escalan automáticamente
remainder=Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
]))


## Paso 4 — Entrenamiento con búsqueda de hiperparámetros

**Por qué `RandomizedSearchCV` y no `GridSearchCV`?**
`GridSearchCV` prueba TODAS las combinaciones: 3 × 3 × 2 × 3 × 3 = 162 fits.
`RandomizedSearchCV` con `n_iter=10` prueba solo 10 combinaciones aleatorias.
En examen de tiempo limitado, 10 iteraciones tardan ~2 min; 162 tardarían ~30 min.

**Por qué `scoring="neg_root_mean_squared_error"`?**
sklearn maximiza métricas, pero queremos minimizar el error.
El prefijo `neg_` invierte el signo: maximizar −RMSE = minimizar RMSE.

**Por qué `cv=3` y no `cv=5`?**
3 folds = 3× más rápido que 5. En examen priorizamos velocidad.
Con datos grandes (16k filas train), 3 folds es suficiente.

**¿Cuándo usar `n_jobs=-1`?**
Usa todos los núcleos de CPU en paralelo. Seguro en examen; puede dar warnings en Jupyter pero funciona.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

# Pipeline completo: preprocesamiento + modelo
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
])

# Espacio de búsqueda de hiperparámetros
# Notación: "nombre_paso__hiperparametro"
param_grid = {
    "preprocessor__geo__n_clusters": [5, 10, 15],       # zonas geográficas
    "preprocessor__geo__gamma":      [0.1, 1.0, 10.0],  # anchura del kernel RBF
    "regressor__n_estimators":       [100, 200],         # número de árboles
    "regressor__max_depth":          [None, 16, 24],     # profundidad máxima
    "regressor__min_samples_leaf":   [1, 3, 5],          # mínimo de muestras por hoja
}

search = RandomizedSearchCV(
    model,
    param_distributions=param_grid,
    n_iter=10,          # probar 10 combinaciones aleatorias
    cv=3,               # validación cruzada de 3 folds
    scoring="neg_root_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

search.fit(X_train, y_train)
print("Mejores parámetros:", search.best_params_)
print(f"Mejor RMSE en CV: {-search.best_score_:,.0f} $")


## Paso 5 — Evaluación final

**Regla de oro: evaluar en test UNA SOLA VEZ al final.**
Si evaluamos en test varias veces para ajustar el modelo, el test se "contamina" y las métricas son optimistas.
Todo el ajuste (GridSearch, elección de features, etc.) debe hacerse con cross-validation sobre train.


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("=" * 40)
print("EVALUACIÓN FINAL EN TEST SET")
print("=" * 40)
print(f"RMSE: {rmse:>12,.0f} $ (error típico)")
print(f"MAE:  {mae:>12,.0f} $ (error mediano)")
print(f"R²:   {r2:>12.4f}   (varianza explicada)")


## Explicación para el examen (en voz alta)

> *"Usé un Pipeline de sklearn para evitar data leakage: el fit solo ocurre sobre train.
> Creé ratios de habitaciones porque los totales absolutos dependen del tamaño del barrio, no de la vivienda.
> Apliqué log a variables sesgadas para acercarlas a distribución normal.
> El clustering geográfico con KMeans + kernel RBF convierte lat/lon en una feature de similitud zonal.
> Ajusté hiperparámetros con RandomizedSearchCV × 3-fold CV para ser eficiente en tiempo.
> Evalué en test una sola vez al final."*

## Problemas frecuentes y soluciones

| Problema | Causa | Solución |
|----------|-------|---------|
| `FileNotFoundError` en el CSV | Ruta relativa incorrecta | Usar `Path("../proyects/COMPLETOS/california-e2e/data/housing.csv")` |
| RMSE muy alto (>60,000) | Sin feature engineering | Añadir ratios y log transform |
| Warnings de KMeans | `n_init` por defecto bajo | Añadir `n_init=10` en `KMeans` |
| `Pipeline` no acepta `feature_names_out` | Versión de sklearn < 1.0 | Actualizar sklearn o quitar el argumento |
| `ColumnTransformer` da error en `set_params` | El transformer no es `BaseEstimator` | Heredar de `BaseEstimator, TransformerMixin` |
